In [1]:
import pandas as pd

from datapreparation.data_preparation import Data_Preparation

path = "../mockdata/"

df_waterings = pd.read_csv(f'{path}waterings.csv')
df_sensors = pd.read_csv(f'{path}sensor_datas.csv')
df_plants = pd.read_csv(f'{path}plants.csv')

df_waterings

,PlantMAC,PredictedFutureWaterTime,LastWaterTime,WaterLevel,PumpTimeInSeconds
0,34:2c:d8:10:0f:2f,2025-01-04T12:00:00,2025-01-01T12:00:00,100.00,10
1,34:2c:d8:10:0f:2f,2025-01-05T11:09:59.546910,2025-01-01T17:00:00,73.69,30
2,34:2c:d8:10:0f:2f,2025-01-06T12:26:30.950313,2025-01-02T12:00:00,79.12,27
3,34:2c:d8:10:0f:2f,2025-01-05T17:23:45.836219,2025-01-03T13:00:00,64.03,24
4,34:2c:d8:10:0f:2f,2025-01-06T08:31:13.149845,2025-01-04T08:00:00,90.07,25
...,...,...,...,...,...
698,d8:00:82:b6:d5:e0,2025-03-21T16:25:06.204993,2025-03-18T07:00:00,75.88,26
699,d8:00:82:b6:d5:e0,2025-03-23T14:31:48.803788,2025-03-18T18:00:00,79.60,23
700,d8:00:82:b6:d5:e0,2025-03-23T01:26:29.733923,2025-03-19T11:00:00,80.41,26
701,d8:00:82:b6:d5:e0,2025-03-24T06:56:10.359391,2025-03-21T11:00:00,92.10,35


In [2]:
mac_map = {mac: i for i, mac in enumerate(df_plants['MAC'].unique())}

df_waterings = df_waterings.drop(columns=['PredictedFutureWaterTime'])
df_waterings = df_waterings.rename(columns={'LastWaterTime': 'Timestamp'})

df_sensors['Timestamp']   = pd.to_datetime(df_sensors['Timestamp'])
df_waterings['Timestamp'] = pd.to_datetime(df_waterings['Timestamp'], format='mixed')

df_sensors['mac_id']   = df_sensors['PlantMAC'].map(mac_map)
df_waterings['mac_id'] = df_waterings['PlantMAC'].map(mac_map)
df_plants['mac_id']    = df_plants['MAC'].map(mac_map)

df_plants_cleaned = df_plants.drop(columns=['MAC', 'Username', 'Name'])

sensors_sorted   = df_sensors.sort_values('Timestamp').reset_index(drop=True).drop(columns=['PlantMAC'])
waterings_sorted = df_waterings.sort_values('Timestamp').reset_index(drop=True).drop(columns=['PlantMAC'])

df_water_sensors = pd.merge_asof(
    sensors_sorted,
    waterings_sorted[['mac_id', 'Timestamp', 'PumpTimeInSeconds','WaterLevel']].rename(columns={'Timestamp': 'LastWaterTimestamp'}),
    left_on='Timestamp',
    right_on='LastWaterTimestamp',
    by='mac_id',
    direction='backward'
)

df_water_sensors['seconds_since_watering'] = (
    df_water_sensors['Timestamp'] - df_water_sensors['LastWaterTimestamp']
).dt.total_seconds()

df_water_sensors = df_water_sensors.drop(columns=['LastWaterTimestamp', 'Timestamp'])

# Bring in plant optimal values
df_water_sensors = df_water_sensors.merge(df_plants_cleaned, on='mac_id', how='left')


df_water_sensors

,Temperature,AirHumidity,SoilHumidity,LightIntensity,mac_id,PumpTimeInSeconds,WaterLevel,seconds_since_watering,OptimalTemperature,OptimalAirHumidity,OptimalSoilHumidity,OptimalLightIntensity
0,23.64,45.55,62.13,418.42,0,10,100.00,0.0,23.1,40.8,58.3,411.6
1,24.43,55.89,64.63,610.03,8,10,100.00,0.0,25.7,62.4,67.0,673.7
2,20.47,71.06,52.24,417.98,7,10,100.00,0.0,22.8,67.2,55.9,422.3
3,23.59,51.47,51.39,538.78,9,10,100.00,0.0,19.6,52.7,60.1,543.2
4,19.17,54.76,62.04,340.83,6,10,100.00,0.0,20.1,58.9,65.2,307.8
...,...,...,...,...,...,...,...,...,...,...,...,...
29995,20.72,33.62,48.41,547.89,1,34,98.25,18000.0,22.6,40.1,51.9,567.8
29996,20.75,49.92,61.38,442.05,12,32,73.85,10800.0,22.3,47.2,60.9,383.9
29997,24.07,43.74,54.93,446.14,0,26,74.71,86400.0,23.1,40.8,58.3,411.6
29998,17.06,46.90,70.00,543.10,9,27,98.85,21600.0,19.6,52.7,60.1,543.2


In [3]:
for mac, group in df_water_sensors.groupby('mac_id'):
    pass

X = df_water_sensors.drop(columns=['mac_id', 'PumpTimeInSeconds'])
y = df_water_sensors['PumpTimeInSeconds']
X

,Temperature,AirHumidity,SoilHumidity,LightIntensity,WaterLevel,seconds_since_watering,OptimalTemperature,OptimalAirHumidity,OptimalSoilHumidity,OptimalLightIntensity
0,23.64,45.55,62.13,418.42,100.00,0.0,23.1,40.8,58.3,411.6
1,24.43,55.89,64.63,610.03,100.00,0.0,25.7,62.4,67.0,673.7
2,20.47,71.06,52.24,417.98,100.00,0.0,22.8,67.2,55.9,422.3
3,23.59,51.47,51.39,538.78,100.00,0.0,19.6,52.7,60.1,543.2
4,19.17,54.76,62.04,340.83,100.00,0.0,20.1,58.9,65.2,307.8
...,...,...,...,...,...,...,...,...,...,...
29995,20.72,33.62,48.41,547.89,98.25,18000.0,22.6,40.1,51.9,567.8
29996,20.75,49.92,61.38,442.05,73.85,10800.0,22.3,47.2,60.9,383.9
29997,24.07,43.74,54.93,446.14,74.71,86400.0,23.1,40.8,58.3,411.6
29998,17.06,46.90,70.00,543.10,98.85,21600.0,19.6,52.7,60.1,543.2
